# 06 · Model expansion

Round 1 compared 4 algorithms (`linear_regression`, `random_forest`, `xgboost`, `lightgbm`)
on the frozen `historical_lead` feature set — XGBoost won (test R² **0.5924**).

This notebook tests a **second round of model families** under the **exact same protocol**
(same time split ≤2016 / 2017–2021 / ≥2022, encoders on train only, `log1p` target,
same metrics, same logging):

| model | family | notes |
|---|---|---|
| `ridge` / `lasso` / `elasticnet` | regularized linear | imputed + scaled |
| `svr` | kernel (RBF) | imputed + scaled |
| `knn` | instance-based | imputed + scaled |
| `mlp` | neural net | imputed + scaled |
| `histgb` | sklearn gradient boosting | — |
| `catboost` | gradient boosting (ordered) | val early-stopping |

Plus a validation-weighted **ensemble** of the tree boosters and a small **XGBoost tuning**
scan. Every result is appended to `reports/experiments/experiment_log.csv`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

from scripts.prepare_dataset import prepare_ml_dataset
from scripts.evaluation import (
    EXTRA_ALGORITHMS,
    FEATURE_SETS,
    TRAIN_END_YEAR,
    VAL_END_YEAR,
    _quiet_fit,
    _split_matrices,
    blend_algorithms,
    compare_extra_algorithms,
    evaluate_predictions,
    log_experiment,
)

ds = prepare_ml_dataset()
print('rows:', len(ds), '| release_years:', ds['release_year'].min(), '-', ds['release_year'].max())

rows: 8040 | release_years: 1925.0 - 2027.0


### Round-2 model comparison

`compare_extra_algorithms` mirrors `compare_algorithms` from notebook 04 — same split,
same matrices, same evaluation, and each run is logged as `alg_<model>`.

In [2]:
extra = compare_extra_algorithms(ds, 'historical_lead', notes='round-2 model expansion')
extra.round(4).sort_values('r2', ascending=False)

,model,mae,rmse,r2,raw_rmse,raw_mae
7,catboost,0.8147,1.0501,0.5887,1.782741e+08,7.244997e+07
1,lasso,0.8381,1.0632,0.5784,1.896394e+08,7.595847e+07
2,elasticnet,0.8381,1.0632,0.5784,1.896554e+08,7.597297e+07
0,ridge,0.8384,1.0636,0.5781,1.896863e+08,7.600073e+07
6,histgb,0.8335,1.0719,0.5715,1.738858e+08,7.311620e+07
3,svr,0.9535,1.2111,0.4529,2.043349e+08,8.737068e+07
4,knn,0.9667,1.2309,0.4349,2.058141e+08,8.026044e+07
5,mlp,1.1276,1.4565,0.2088,1.096964e+09,2.289174e+08


### Validation-weighted ensemble

Average held-out predictions of all the tree models, weighting each member by its
**validation** R² — the test split is untouched during weight selection.

In [3]:
ens = blend_algorithms(
    ds, 'historical_lead',
    ['xgboost', 'lightgbm', 'catboost', 'histgb', 'random_forest'],
    notes='tree-ensemble blend',
)
print('weights:', ens['weights'])
pd.Series(ens['test_metrics']).round(4)

/Users/hariz/Desktop/TMDB-movie-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


weights: {'xgboost': np.float64(0.202), 'lightgbm': np.float64(0.2), 'catboost': np.float64(0.202), 'histgb': np.float64(0.2), 'random_forest': np.float64(0.196)}


mae         8.164000e-01
rmse        1.049700e+00
r2          5.890000e-01
raw_rmse    1.749937e+08
raw_mae     7.163325e+07
dtype: float64

### XGBoost tuning scan

Four hyperparameter combinations, selected on validation R², then re-evaluated on test.

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = _split_matrices(ds, FEATURE_SETS['historical_lead'])

grid = [
    dict(learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8),
    dict(learning_rate=0.05, max_depth=6, subsample=0.9, colsample_bytree=0.8),  # round-1 default
    dict(learning_rate=0.08, max_depth=8, subsample=0.8, colsample_bytree=0.7),
    dict(learning_rate=0.03, max_depth=8, subsample=0.9, colsample_bytree=0.7),
]

rows = []
fits = {}
for g in grid:
    m = XGBRegressor(
        n_estimators=500, random_state=42, verbosity=0, early_stopping_rounds=50, **g
    )
    _quiet_fit(m, X_train, y_train, X_val, y_val)
    val_r2 = r2_score(y_val, m.predict(X_val))
    test_r2 = r2_score(y_test, m.predict(X_test))
    fits[tuple(sorted(g.items()))] = m
    rows.append((g, round(val_r2, 4), round(test_r2, 4)))

scan = pd.DataFrame(rows, columns=['params', 'val_r2', 'test_r2'])
scan['params'] = scan['params'].astype(str)
scan.sort_values('val_r2', ascending=False)

,params,val_r2,test_r2
0,"{'learning_rate': 0.03, 'max_depth': 4, 'subsa...",0.6317,0.5913
1,"{'learning_rate': 0.05, 'max_depth': 6, 'subsa...",0.6301,0.5945
3,"{'learning_rate': 0.03, 'max_depth': 8, 'subsa...",0.6281,0.5836
2,"{'learning_rate': 0.08, 'max_depth': 8, 'subsa...",0.6162,0.5742


In [5]:
best_params = rows[np.argmax([r[1] for r in rows])][0]
print('best on val:', best_params)
best_model = fits[tuple(sorted(best_params.items()))]

mets = evaluate_predictions(y_test, best_model.predict(X_test))
log_experiment({
    'experiment_id': 'alg_xgboost_tuned',
    'feature_set': 'historical_lead',
    'model': 'xgboost_tuned',
    'hyperparameters': str(best_params),
    'train_period': f'<={TRAIN_END_YEAR}',
    'validation_period': f'{TRAIN_END_YEAR + 1}-{VAL_END_YEAR}',
    'test_period': f'>={VAL_END_YEAR + 1}',
    **{k: round(v, 4) if k in ('mae', 'rmse', 'r2') else round(v, 0) for k, v in mets.items()},
    'notes': 'selected on val R2 from 4-combo scan',
})
print(pd.Series(mets).round(4))

best on val: {'learning_rate': 0.03, 'max_depth': 4, 'subsample': 0.8, 'colsample_bytree': 0.8}
mae         8.109000e-01
rmse        1.046800e+00
r2          5.913000e-01
raw_rmse    1.733476e+08
raw_mae     7.162824e+07
dtype: float64


### Final ranking

All round-1 + round-2 + tuned + ensemble results, plus the round-1 champion from notebook 04.

In [6]:
from scripts.evaluation import read_experiment_log
log = read_experiment_log()
log.index = [f'{r} · {log.loc[r,"model"]}' for r in log.index]
tab = (log[log['feature_set'] == 'historical_lead']
       .drop(columns=['feature_set', 'train_period', 'validation_period', 'test_period'])
       .sort_values('r2', ascending=False))
tab.drop_duplicates('experiment_id').round(4)

,experiment_id,model,hyperparameters,mae,rmse,r2,raw_rmse,raw_mae,notes
7 · XGBRegressor,historical_lead,XGBRegressor,"{'n_estimators': 500, 'max_depth': 6, 'learnin...",0.8130,1.0453,0.5924,1.832165e+08,71512857.0,incremental feature stage
11 · xgboost,alg_xgboost,xgboost,"{'n_estimators': 500, 'max_depth': 6, 'learnin...",0.8130,1.0453,0.5924,1.832165e+08,71512857.0,algorithm comparison on frozen feature set
22 · xgboost_tuned,alg_xgboost_tuned,xgboost_tuned,"{'learning_rate': 0.03, 'max_depth': 4, 'subsa...",0.8109,1.0468,0.5913,1.733476e+08,71628242.0,selected on val R2 from 4-combo scan
21 · weighted_blend,alg_ensemble,weighted_blend,"val-R2-weighted blend of [xgboost, lightgbm, c...",0.8164,1.0497,0.5890,1.749937e+08,71633250.0,"tree-ensemble blend weights: xgboost:0.20, lig..."
20 · catboost,alg_catboost,catboost,"CatBoost(iters=1000,lr=0.05,depth=6), early-st...",0.8147,1.0501,0.5887,1.782741e+08,72449974.0,round-2 model expansion
14 · lasso,alg_lasso,lasso,"Lasso(alpha=1e-3), imputed+scaled",0.8381,1.0632,0.5784,1.896394e+08,75958474.0,round-2 model expansion
15 · elasticnet,alg_elasticnet,elasticnet,"ElasticNet(alpha=1e-3,l1_ratio=0.5), imputed+s...",0.8381,1.0632,0.5784,1.896554e+08,75972971.0,round-2 model expansion
9 · linear_regression,alg_linear_regression,linear_regression,SimpleImputer(median)+StandardScaler+LinearReg...,0.8381,1.0633,0.5783,1.896525e+08,75961470.0,algorithm comparison on frozen feature set
13 · ridge,alg_ridge,ridge,"Ridge(alpha=10), imputed+scaled",0.8384,1.0636,0.5781,1.896863e+08,76000729.0,round-2 model expansion
12 · lightgbm,alg_lightgbm,lightgbm,"n_estimators=500,lr=0.05,subsample=0.9,colsamp...",0.8416,1.0719,0.5715,1.760893e+08,73552286.0,algorithm comparison on frozen feature set


### Verdict

**Round-2 does not beat round-1's XGBoost (test R² 0.5924).** The production model is unchanged.

| approach | test R² | takeaway |
|---|---|---|
| `catboost` | 0.5887 | best challenger, still below XGBoost |
| `lasso` / `elasticnet` / `ridge` | 0.578 | the linear ceiling — regularized linear ≈ plain linear on these features |
| `histgb` | 0.5715 | ~parity with LightGBM/RF |
| `svr` / `knn` | 0.45 / 0.43 | kernel & instance methods are poor on noisy, log-heavy movie revenue |
| `mlp` | 0.21 | heavily underfits on ~6.3k rows × noisy target |
| ensemble (val-R² blend of 5 trees) | 0.5890 | averaging a strong XGBoost with weaker members pulls it *down* (all val R² ≈0.20 → near-equal weights) |
| val-tuned XGBoost | 0.5913 | val-picked (lr 0.03, depth 4) is slightly *below* the robust default 0.5924 — the round-1 config stays |

**Decision:** XGBoost with the round-1 hyperparameters remains the frozen production model
(artifacts in `models/` are untouched). All 10 new runs are logged in
`reports/experiments/experiment_log.csv`.